In [1]:
!pip -q install langchain-text-splitters groq chromadb gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [2]:
from datasets import load_dataset

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import chromadb
from groq import Groq
import gradio as gr

In [3]:
dataset = load_dataset("Jr23xd23/ArabicText-Large", split="train[:30000]")
dataset = dataset["text"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/251M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/173M [00:00<?, ?B/s]

data/train-00002-of-00006.parquet:   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00003-of-00006.parquet:   0%|          | 0.00/156M [00:00<?, ?B/s]

data/train-00004-of-00006.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

data/train-00005-of-00006.parquet:   0%|          | 0.00/149M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/743288 [00:00<?, ? examples/s]

In [4]:
len(dataset)

30000

In [5]:
embd_model = "Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2"
model = SentenceTransformer(embd_model)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/637 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

# chunk functions

In [6]:
def fixed_chunking(text):

    splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=500,
        chunk_overlap=50,
    )

    return splitter.split_text(text)

In [7]:
def recursive_chunking(text):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", " "]
    )

    return splitter.split_text(text)

In [8]:
def semantic_chunking(text, threshold=0.5):

    sentences = text.split(".")
    sentences = [s.strip() for s in sentences if s.strip()]

    embeddings = model.encode(sentences)

    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):

        similarity = cosine_similarity(
            [embeddings[i - 1]],
            [embeddings[i]]
        )[0][0]

        if similarity >= threshold:
            current_chunk.append(sentences[i])

        else:
            chunks.append(". ".join(current_chunk))
            current_chunk = [sentences[i]]

    chunks.append(". ".join(current_chunk))

    return chunks

In [9]:
all_chunks = []

sample_dataset = dataset[:100]

for text in sample_dataset:

    # chunks = fixed_chunking(text)
    # chunks = recursive_chunking(text)
    chunks = semantic_chunking(text)

    all_chunks.extend(chunks)

len(all_chunks)

4148

In [10]:
embeddings = model.encode(
    all_chunks,
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/130 [00:00<?, ?it/s]

In [11]:
client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_or_create_collection(name="ar_wiki")

collection.add(
    documents = all_chunks,
    embeddings = embeddings.tolist(),
    ids=[str(i) for i in range(len(all_chunks))]
)

In [12]:
def search(query, n_results=1):

    query_embedding = model.encode([query])[0]

    results = collection.query(
        query_embeddings= [query_embedding.tolist()],
        n_results = n_results
    )

    out = results["documents"][0]

    return out

In [14]:
client_g = Groq(api_key="...")

def ask_rag_g(query):
    docs = search(query, 5)
    context = "\n\n".join(docs)

    prompt = f"""
النص المرجعي:
{context}

السؤال:
{query}
"""

    response = client_g.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "اجب على السؤال بناء على المعلومات المعطاة فقط، لا تقم باختصار الاجابه. لا تقم بالتعليق "},
            {"role": "user", "content": prompt}
        ],
        temperature=0.5
    )

    answer = response.choices[0].message.content

    return answer

In [48]:
print(ask_rag_g("فوائد الماء"))

تتضمن الفوائد المائية العديد من الجوانب، منها:

- يعد الماء مصدر حيوي للنباتات والحيوانات، حيث يسهم في نمو النباتات وتأمين مياه شرب للحيوانات.

- يستغل العديد من الكائنات الحية خواص الماء، مثل الحشرات والعنكبيات، التي تستغل خاصية التوتر السطحي للماء في حياتها اليومية بشكل كبير.

- يمثل الماء جزءاً كبيراً من جسم الإنسان، حيث تتراوح نسبة الماء في الجسم بين 55% إلى 78%، مما يبرز الأهمية الحيوية له بالنسبة لبقاء البشرية.

- يلعب الماء دوراً هاماً في أمور النظافة، حيث يستخدم للاستحمام وغسل الملابس وتنظيف المنازل، بالإضافة إلى جلي الأواني.

- يعد الماء أساسياً للطبخ، حيث يستخدم بطرق مختلفة من أجل تحضير الطعام.

- يلعب الماء دوراً هاماً في المجتمعات البشرية، حيث يستخدم بشكل رئيسي ك مصدر آمن لمياه الشرب، بالإضافة إلى استخدامه في قضاء الحاجات المنزلية الأساسية.

- يستخدم الماء بشكل أساسي في الزراعة، وخاصة من أجل الري، وكذلك في الصناعة وخاصة صناعة الغذاء.

- يعد الماء وسطا حيويا يسمح بقيام تفاعلات عضوية حيوية تؤدي في النهاية إلى تأمين التناسخ الذاتي، مما يضمن استمرار التناسل وبقاء الكائنات الحية

In [18]:
demo = gr.Interface(
    fn = ask_rag_g,
    inputs=gr.Textbox(
        lines=2,
        placeholder="اكتب سؤالك هنا...",
        label="السؤال"
    ),
    outputs=gr.Textbox(
        lines=10,
        label="الإجابة"
    ),
    title="Arabic RAG System",
    allow_flagging="never"
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0d5ba143b3c4970f79.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
